##### Imports

In [1]:
# pip install llama-index-lms-ollama
# pip install llama-index-embeddings-ollama
import os
import shutil
from pathlib import Path
import csv
import json
import re
from collections import defaultdict
from difflib import SequenceMatcher
import tiktoken
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

/home/cpanagiotop/CaseBundleGen_Eval/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### Preprocessing

In [2]:
cases_dir = Path("cases")
quesnas_dir = Path("quesNAs")

# Get all case folders
case_folders = sorted([d for d in cases_dir.iterdir() if d.is_dir() and d.name.startswith("case_")])

print(f"Processing {len(case_folders)} case folders...\n")

for case_folder in case_folders:
    case_name = case_folder.name
    
    # Delete metadata.json if it exists
    metadata_file = case_folder / "metadata.json"
    if metadata_file.exists():
        metadata_file.unlink()
    
    # Move qa.json to quesNAs folder
    qa_file = case_folder / "qa.json"
    if qa_file.exists():
        # Create quesNAs/case_* folder if it doesn't exist
        quesnas_case_dir = quesnas_dir / case_name
        quesnas_case_dir.mkdir(parents=True, exist_ok=True)
        
        # Move qa.json (this will replace if it already exists)
        dest_qa_file = quesnas_case_dir / "qa.json"
        shutil.copy2(qa_file, dest_qa_file)
        qa_file.unlink()

print("✓ Preparation complete! Deleted metadata.json and moved qa.json files to quesNAs/")

Processing 100 case folders...

✓ Preparation complete! Deleted metadata.json and moved qa.json files to quesNAs/


##### Preparation of evaluation functions


In [7]:
EMBED_DEVICE = os.getenv("EMBED_DEVICE", "cpu")
EMBED_BATCH_SIZE = int(os.getenv("EMBED_BATCH_SIZE", "8"))
INDEX_INSERT_BATCH_SIZE = int(os.getenv("INDEX_INSERT_BATCH_SIZE", "16"))

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-base-en-v1.5",
    device=EMBED_DEVICE,
    embed_batch_size=EMBED_BATCH_SIZE,
)
Settings.llm = Ollama(model="gemma3:12b", request_timeout=360.0)

print(
    f"Embedding device={EMBED_DEVICE}, embed_batch_size={EMBED_BATCH_SIZE}, "
    f"index_insert_batch_size={INDEX_INSERT_BATCH_SIZE}"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15592.79it/s]


Embedding device=cpu, embed_batch_size=8, index_insert_batch_size=16


In [8]:
import gc

try:
    import torch
except ImportError:
    torch = None


# Encourage short, directly comparable outputs from the query model.
STRICT_ANSWER_SUFFIX = (
    "\n\nReturn only the final answer. No explanation. "
    "If the answer is unavailable in the case files, return exactly: None."
)


def clear_runtime_memory():
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()


def normalize_text(text: str) -> str:
    if text is None:
        return ""
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text


def exact_match(reference: str, prediction: str) -> bool:
    return normalize_text(reference) == normalize_text(prediction)


def fuzzy_match(reference: str, prediction: str, threshold: float = 0.80) -> bool:
    ref_norm = normalize_text(reference)
    pred_norm = normalize_text(prediction)
    if not ref_norm or not pred_norm:
        return False
    return SequenceMatcher(None, ref_norm, pred_norm).ratio() >= threshold


def response_to_text(response) -> str:
    if response is None:
        return ""
    if hasattr(response, "text"):
        return response.text
    if hasattr(response, "response"):
        return response.response
    return str(response)


def list_case_ids():
    return sorted([p.name for p in Path("cases").iterdir() if p.is_dir()])


def load_qa(case_id: str):
    qa_path = Path("quesNAs") / case_id / "qa.json"
    with qa_path.open("r", encoding="utf-8") as f:
        return json.load(f)


def build_case_index(case_id: str, insert_batch_size: int | None = None):
    case_path = Path("cases") / case_id
    documents = SimpleDirectoryReader(str(case_path)).load_data()
    batch_size = insert_batch_size or INDEX_INSERT_BATCH_SIZE
    return VectorStoreIndex.from_documents(documents, insert_batch_size=batch_size)


def judge_answer(question: str, expected: str, predicted: str, llm=None) -> tuple[str, str, str]:
    if llm is None:
        llm = Settings.llm
    prompt = (
        "You are a precise QA judge. Compare the expected answer and the model answer "
        "for the question below. Return only one word: EXACT, PARTIAL, or WRONG. "
        "Do not add any explanation, punctuation, or extra text.\n\n"
        f"Question: {question}\n"
        f"Expected answer: {expected}\n"
        f"Model answer: {predicted}\n"
    )
    response = llm.complete(prompt=prompt)
    raw_text = response_to_text(response).strip()
    normalized_label = normalize_text(raw_text).upper()
    if normalized_label == "EXACT":
        return "EXACT", raw_text, normalized_label
    if normalized_label == "PARTIAL":
        return "PARTIAL", raw_text, normalized_label
    if normalized_label == "WRONG":
        return "WRONG", raw_text, normalized_label
    if "exact" in normalized_label:
        return "EXACT", raw_text, normalized_label
    if "partial" in normalized_label:
        return "PARTIAL", raw_text, normalized_label
    if "wrong" in normalized_label:
        return "WRONG", raw_text, normalized_label
    return "WRONG", raw_text, normalized_label


def evaluate_case(case_id: str, index, use_judge: bool = True):
    qa_pairs = load_qa(case_id)
    query_engine = index.as_query_engine()

    case_results = []
    for pair in qa_pairs:
        question = pair.get("question")
        expected = pair.get("answer")

        # Force short format at query time but score raw output directly.
        constrained_question = f"{question}{STRICT_ANSWER_SUFFIX}"
        response = query_engine.query(constrained_question)
        predicted = response_to_text(response)

        is_exact = exact_match(expected, predicted)
        is_similar = fuzzy_match(expected, predicted)
        judge_label, judge_raw, judge_normalized = (
            judge_answer(question, expected, predicted) if use_judge else (None, None, None)
        )

        case_results.append(
            {
                "case_id": case_id,
                "question": question,
                "expected_answer": expected,
                "predicted_answer": predicted,
                "exact_match": is_exact,
                "similar_match": is_similar,
                "judge_label": judge_label,
                "judge_raw": judge_raw,
                "judge_normalized": judge_normalized,
            }
        )

    return case_results


def evaluate_all_cases(
    use_judge: bool = True,
    case_ids: list[str] | None = None,
    fail_on_oom: bool = True,
):
    all_results = []
    selected_case_ids = case_ids or list_case_ids()

    for idx, case_id in enumerate(selected_case_ids, start=1):
        print(f"[{idx}/{len(selected_case_ids)}] Building index for {case_id}...")
        index = None
        try:
            index = build_case_index(case_id)
            print(f"[{idx}/{len(selected_case_ids)}] Evaluating {case_id}...")
            case_results = evaluate_case(case_id, index, use_judge=use_judge)
            all_results.extend(case_results)
        except RuntimeError as e:
            is_oom = "out of memory" in str(e).lower()
            if is_oom:
                print(f"OOM while processing {case_id}: {e}")
                clear_runtime_memory()
                if fail_on_oom:
                    raise
                print(f"Skipping {case_id} and continuing...")
                continue
            raise
        finally:
            del index
            clear_runtime_memory()

    return all_results


def summarize_results(results):
    total = len(results)
    exact = sum(1 for r in results if r["exact_match"])
    similar = sum(1 for r in results if r["similar_match"])
    judge_exact = sum(1 for r in results if r.get("judge_label") == "EXACT")
    print(f"Total questions: {total}")
    print(f"Exact-match accuracy: {exact}/{total} = {exact/total:.2%}")
    print(f"Fuzzy-match accuracy: {similar}/{total} = {similar/total:.2%}")
    print(f"Judge EXACT accuracy: {judge_exact}/{total} = {judge_exact/total:.2%}")
    return {
        "total_questions": total,
        "exact_match_count": exact,
        "fuzzy_match_count": similar,
        "judge_exact_count": judge_exact,
        "exact_match_accuracy": exact / total if total else 0.0,
        "fuzzy_match_accuracy": similar / total if total else 0.0,
        "judge_exact_accuracy": judge_exact / total if total else 0.0,
    }


def summarize_by_case(results):
    cases = defaultdict(list)
    for r in results:
        cases[r["case_id"]].append(r)

    per_case = {}
    for case_id, rows in cases.items():
        total = len(rows)
        exact = sum(1 for r in rows if r["exact_match"])
        similar = sum(1 for r in rows if r["similar_match"])
        judge_exact = sum(1 for r in rows if r.get("judge_label") == "EXACT")
        per_case[case_id] = {
            "total_questions": total,
            "exact_match_count": exact,
            "fuzzy_match_count": similar,
            "judge_exact_count": judge_exact,
            "exact_match_accuracy": exact / total if total else 0.0,
            "fuzzy_match_accuracy": similar / total if total else 0.0,
            "judge_exact_accuracy": judge_exact / total if total else 0.0,
        }

    exact_rates = [stats["exact_match_accuracy"] for stats in per_case.values()]
    fuzzy_rates = [stats["fuzzy_match_accuracy"] for stats in per_case.values()]
    judge_rates = [stats["judge_exact_accuracy"] for stats in per_case.values()]

    if exact_rates:
        print("Per-case exact match rates:")
        for case_id, stats in per_case.items():
            print(f" - {case_id}: {stats['exact_match_accuracy']:.2%}")
        print(f"Min exact-match accuracy: {min(exact_rates):.2%}")
        print(f"Max exact-match accuracy: {max(exact_rates):.2%}")
        print(f"Min fuzzy-match accuracy: {min(fuzzy_rates):.2%}")
        print(f"Max fuzzy-match accuracy: {max(fuzzy_rates):.2%}")
        print(f"Min judge EXACT accuracy: {min(judge_rates):.2%}")
        print(f"Max judge EXACT accuracy: {max(judge_rates):.2%}")

    return {
        "per_case": per_case,
        "min_exact_match_accuracy": min(exact_rates) if exact_rates else 0.0,
        "max_exact_match_accuracy": max(exact_rates) if exact_rates else 0.0,
        "min_fuzzy_match_accuracy": min(fuzzy_rates) if fuzzy_rates else 0.0,
        "max_fuzzy_match_accuracy": max(fuzzy_rates) if fuzzy_rates else 0.0,
        "min_judge_exact_accuracy": min(judge_rates) if judge_rates else 0.0,
        "max_judge_exact_accuracy": max(judge_rates) if judge_rates else 0.0,
    }


def write_results_csv(results, path="evaluation_results.csv"):
    if not results:
        print("No results to write.")
        return
    fieldnames = [
        "case_id",
        "question",
        "expected_answer",
        "predicted_answer",
        "exact_match",
        "similar_match",
        "judge_label",
        "judge_raw",
        "judge_normalized",
    ]
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in results:
            writer.writerow(row)
    print(f"Written results to {path}")


# Example run:
# full_results = evaluate_all_cases(use_judge=True)
# batch_results = evaluate_all_cases(use_judge=True, case_ids=list_case_ids()[:20], fail_on_oom=False)
# summary = summarize_results(full_results)
# case_summary = summarize_by_case(full_results)
# write_results_csv(full_results, "case_benchmark_results.csv")

In [9]:
def get_token_encoding(encoding_name: str = "cl100k_base"):
    try:
        return tiktoken.get_encoding(encoding_name)
    except Exception:
        return tiktoken.get_encoding("cl100k_base")


def count_tokens(text: str, encoding_name: str = "cl100k_base") -> int:
    if not text:
        return 0
    encoding = get_token_encoding(encoding_name)
    return len(encoding.encode(text))


def case_token_stats(case_id: str, encoding_name: str = "cl100k_base") -> dict:
    case_path = Path("cases") / case_id
    documents = SimpleDirectoryReader(str(case_path)).load_data()
    encoding = get_token_encoding(encoding_name)
    token_counts = []
    for doc in documents:
        text = getattr(doc, "text", None)
        if text is None:
            text = str(doc)
        token_counts.append(len(encoding.encode(text or "")))

    total_tokens = sum(token_counts)
    return {
        "case_id": case_id,
        "document_count": len(token_counts),
        "token_count": total_tokens,
        "avg_tokens_per_document": total_tokens / len(token_counts) if token_counts else 0,
    }


def compute_case_token_stats(encoding_name: str = "cl100k_base") -> list[dict]:
    return [case_token_stats(case_id, encoding_name) for case_id in list_case_ids()]


def summarize_token_stats(case_stats: list[dict]) -> dict:
    if not case_stats:
        return {
            "total_cases": 0,
            "total_tokens": 0,
            "min_tokens_per_case": 0,
            "max_tokens_per_case": 0,
            "average_tokens_per_case": 0.0,
        }
    
    total_cases = len(case_stats)
    token_counts_per_case = [stats["token_count"] for stats in case_stats]
    total_tokens = sum(token_counts_per_case)
    avg_per_case = total_tokens / total_cases if total_cases else 0.0
    min_tokens = min(token_counts_per_case)
    max_tokens = max(token_counts_per_case)
    
    print(f"Cases: {total_cases}")
    print(f"Total tokens across all cases: {total_tokens}")
    print(f"Min tokens per case: {min_tokens}")
    print(f"Max tokens per case: {max_tokens}")
    print(f"Average tokens per case: {avg_per_case:.2f}")
    
    return {
        "total_cases": total_cases,
        "total_tokens": total_tokens,
        "min_tokens_per_case": min_tokens,
        "max_tokens_per_case": max_tokens,
        "average_tokens_per_case": avg_per_case,
    }


def write_case_token_stats_csv(case_stats: list[dict], path: str = "case_token_stats.csv"):
    if not case_stats:
        print("No case token stats to write.")
        return
    fieldnames = ["case_id", "document_count", "token_count", "avg_tokens_per_document"]
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in case_stats:
            writer.writerow(row)
    print(f"Written case token stats to {path}")

##### Token count

In [6]:
# Compute case token statistics and write them to CSV.
case_stats = compute_case_token_stats()
token_summary = summarize_token_stats(case_stats)
write_case_token_stats_csv(case_stats, "case_token_stats.csv")
print(token_summary)


Cases: 100
Total tokens across all cases: 174046
Min tokens per case: 1186
Max tokens per case: 2318
Average tokens per case: 1740.46
Written case token stats to case_token_stats.csv
{'total_cases': 100, 'total_tokens': 174046, 'min_tokens_per_case': 1186, 'max_tokens_per_case': 2318, 'average_tokens_per_case': 1740.46}


##### Evaluation and results

In [10]:
# Run the full benchmark, print summaries, and export to CSV.
# fail_on_oom=False skips a case if a rare OOM still occurs.
results = evaluate_all_cases(fail_on_oom=False)
summary = summarize_results(results)
case_summary = summarize_by_case(results)
write_results_csv(results, "case_benchmark_results.csv")
print(summary)
print(case_summary)

[1/100] Building index for case_001...
[1/100] Evaluating case_001...
[2/100] Building index for case_002...
[2/100] Evaluating case_002...
[3/100] Building index for case_003...
[3/100] Evaluating case_003...
[4/100] Building index for case_004...
[4/100] Evaluating case_004...
[5/100] Building index for case_005...
[5/100] Evaluating case_005...
[6/100] Building index for case_006...
[6/100] Evaluating case_006...
[7/100] Building index for case_007...
[7/100] Evaluating case_007...
[8/100] Building index for case_008...
[8/100] Evaluating case_008...
[9/100] Building index for case_009...
[9/100] Evaluating case_009...
[10/100] Building index for case_010...
[10/100] Evaluating case_010...
[11/100] Building index for case_011...
[11/100] Evaluating case_011...
[12/100] Building index for case_012...
[12/100] Evaluating case_012...
[13/100] Building index for case_013...
[13/100] Evaluating case_013...
[14/100] Building index for case_014...
[14/100] Evaluating case_014...
[15/100] B